# 10 - LangGraph y Flujos de Trabajo

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2.5-3 horas

---

## Índice

1. [Introducción a LangGraph](#intro)
2. [Estados y Grafos](#estados)
3. [Flujo RAG con LangGraph](#rag)
4. [Auto-corrección](#correccion)
5. [Checkpoints y Persistencia](#checkpoints)
6. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Crear grafos de estados con LangGraph
- Implementar flujos condicionales
- Añadir auto-corrección a sistemas RAG
- Usar checkpoints para persistencia

<a name="intro"></a>
## 1. Introducción a LangGraph

**LangGraph** es una librería de LangChain para crear flujos de trabajo como grafos de estados.

### ¿Por qué LangGraph?

- **Control explícito**: Define exactamente el flujo
- **Condicionales**: Diferentes caminos según resultados
- **Ciclos**: Permite iteraciones y re-intentos
- **Estado**: Mantiene información entre nodos
- **Persistencia**: Checkpoints para recuperación

In [ ]:
# Install
#!pip install -q langchain langchain-groq langgraph langchain-huggingface faiss-cpu

In [16]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("GROQ API Key: ")

from langchain_groq import ChatGroq
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
print("Configurado ✓")

Configurado ✓


<a name="estados"></a>
## 2. Estados y Grafos

En LangGraph, definimos:
- **State**: Datos que fluyen por el grafo
- **Nodes**: Funciones que procesan el estado
- **Edges**: Conexiones entre nodos

In [2]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

# Define state
class SimpleState(TypedDict):
    messages: List[str]
    current_step: str

# Define nodes
def step_one(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 1 completado"]
    return {"messages": messages, "current_step": "one"}

def step_two(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 2 completado"]
    return {"messages": messages, "current_step": "two"}

def step_three(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 3 completado"]
    return {"messages": messages, "current_step": "three"}

# Build graph
workflow = StateGraph(SimpleState)
workflow.add_node("step_one", step_one)
workflow.add_node("step_two", step_two)
workflow.add_node("step_three", step_three)

# Add edges
workflow.add_edge(START, "step_one")
workflow.add_edge("step_one", "step_two")
workflow.add_edge("step_two", "step_three")
workflow.add_edge("step_three", END)

# Compile
app = workflow.compile()
print("Grafo compilado ✓")

Grafo compilado ✓


In [3]:
# Run the graph
result = app.invoke({"messages": ["Inicio"], "current_step": ""})

print("Resultado:")
for msg in result["messages"]:
    print(f"  - {msg}")

Resultado:
  - Inicio
  - Paso 1 completado
  - Paso 2 completado
  - Paso 3 completado


### Grafos con condicionales

In [4]:
from typing import Literal

class ConditionalState(TypedDict):
    query: str
    query_type: str
    response: str

def classify_query(state: ConditionalState) -> ConditionalState:
    """Classify the query type."""
    query = state["query"].lower()
    if "precio" in query or "costo" in query:
        return {**state, "query_type": "pricing"}
    elif "horario" in query or "hora" in query:
        return {**state, "query_type": "schedule"}
    else:
        return {**state, "query_type": "general"}

def handle_pricing(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Los precios son: Básico 99€, Pro 299€, Enterprise consultar."}

def handle_schedule(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Horario: Lunes a Viernes, 9:00 a 18:00."}

def handle_general(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Para más información, contacta con soporte@empresa.com"}

def route_query(state: ConditionalState) -> Literal["pricing", "schedule", "general"]:
    return state["query_type"]

# Build conditional graph
cond_workflow = StateGraph(ConditionalState)
cond_workflow.add_node("classify", classify_query)
cond_workflow.add_node("pricing", handle_pricing)
cond_workflow.add_node("schedule", handle_schedule)
cond_workflow.add_node("general", handle_general)

cond_workflow.add_edge(START, "classify")
cond_workflow.add_conditional_edges(
    "classify",
    route_query,
    {"pricing": "pricing", "schedule": "schedule", "general": "general"}
)
cond_workflow.add_edge("pricing", END)
cond_workflow.add_edge("schedule", END)
cond_workflow.add_edge("general", END)

cond_app = cond_workflow.compile()
print("Grafo condicional compilado ✓")

Grafo condicional compilado ✓


In [5]:
# Test conditional routing
queries = [
    "¿Cuál es el precio del plan básico?",
    "¿Cuál es el horario de atención?",
    "¿Tienen servicio en México?"
]

for q in queries:
    result = cond_app.invoke({"query": q, "query_type": "", "response": ""})
    print(f"Q: {q}")
    print(f"A: {result['response']}\n")

Q: ¿Cuál es el precio del plan básico?
A: Los precios son: Básico 99€, Pro 299€, Enterprise consultar.

Q: ¿Cuál es el horario de atención?
A: Horario: Lunes a Viernes, 9:00 a 18:00.

Q: ¿Tienen servicio en México?
A: Para más información, contacta con soporte@empresa.com



<a name="rag"></a>
## 3. Flujo RAG con LangGraph

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

# Create vector store
docs = [
    Document(page_content="El IBI se paga anualmente basado en el valor catastral."),
    Document(page_content="El IVTM grava la titularidad de vehículos matriculados."),
    Document(page_content="El ICIO se liquida al finalizar construcciones u obras."),
    Document(page_content="Las bonificaciones pueden reducir hasta un 90% el impuesto."),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vector store creado ✓")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store creado ✓


In [7]:
from typing import List
from langchain_core.messages import BaseMessage

class RAGState(TypedDict):
    messages: List[BaseMessage]
    context: str
    response: str

def retrieve_context(state: RAGState) -> RAGState:
    """Retrieve relevant documents."""
    query = state["messages"][-1].content
    docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate_response(state: RAGState) -> RAGState:
    """Generate response using LLM."""
    query = state["messages"][-1].content
    context = state["context"]
    
    prompt = f"""Responde basándote en el contexto.
    
Contexto: {context}

Pregunta: {query}

Respuesta:"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

# Build RAG graph
rag_workflow = StateGraph(RAGState)
rag_workflow.add_node("retrieve", retrieve_context)
rag_workflow.add_node("generate", generate_response)

rag_workflow.add_edge(START, "retrieve")
rag_workflow.add_edge("retrieve", "generate")
rag_workflow.add_edge("generate", END)

rag_app = rag_workflow.compile()
print("RAG graph compilado ✓")

RAG graph compilado ✓


In [8]:
# Test RAG
result = rag_app.invoke({
    "messages": [HumanMessage(content="¿Qué es el IBI?")],
    "context": "",
    "response": ""
})

print(f"Respuesta: {result['response']}")

Respuesta: El IBI (Impuesto de Bienes Inmuebles) es un impuesto que se paga anualmente y se basa en el valor catastral de un inmueble.


<a name="correccion"></a>
## 4. Auto-corrección

Añadimos un paso de verificación y corrección.

In [10]:
class CorrectionState(TypedDict):
    query: str
    context: str
    response: str
    corrected_response: str
    needs_correction: bool

def retrieve(state: CorrectionState) -> CorrectionState:
    docs = retriever.invoke(state["query"])
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate(state: CorrectionState) -> CorrectionState:
    prompt = f"Contexto: {state['context']}\nPregunta: {state['query']}\nRespuesta:"
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

def check_response(state: CorrectionState) -> CorrectionState:
    """Check if response needs correction."""
    check_prompt = f"""¿La siguiente respuesta está basada en el contexto?
    
Contexto: {state['context']}
Respuesta: {state['response']}

Responde solo 'SI' o 'NO'."""
    
    check = llm.invoke(check_prompt)
    needs_correction = "NO" in check.content.upper()
    return {**state, "needs_correction": needs_correction}

def correct_response(state: CorrectionState) -> CorrectionState:
    """Correct the response."""
    correct_prompt = f"""Mejora esta respuesta basándote solo en el contexto.
    
Contexto: {state['context']}
Respuesta original: {state['response']}

Respuesta mejorada:"""
    
    corrected = llm.invoke(correct_prompt)
    return {**state, "corrected_response": corrected.content}

def route_correction(state: CorrectionState) -> Literal["correct", "end"]:
    return "correct" if state["needs_correction"] else "end"

# Build correction graph
corr_workflow = StateGraph(CorrectionState)
corr_workflow.add_node("retrieve", retrieve)
corr_workflow.add_node("generate", generate)
corr_workflow.add_node("check", check_response)
corr_workflow.add_node("correct", correct_response)

corr_workflow.add_edge(START, "retrieve")
corr_workflow.add_edge("retrieve", "generate")
corr_workflow.add_edge("generate", "check")
corr_workflow.add_conditional_edges("check", route_correction, {"correct": "correct", "end": END})
corr_workflow.add_edge("correct", END)

corr_app = corr_workflow.compile()
print("Grafo con corrección compilado ✓")

Grafo con corrección compilado ✓


In [11]:
# Test
result = corr_app.invoke({
    "query": "¿Cuándo se paga el IVTM?",
    "context": "",
    "response": "",
    "corrected_response": "",
    "needs_correction": False
})

print(f"Respuesta original: {result['response']}")
print(f"Necesitó corrección: {result['needs_correction']}")
if result['corrected_response']:
    print(f"Respuesta corregida: {result['corrected_response']}")

Respuesta original: El IVTM se paga anualmente, al igual que el IBI, pero en este caso, se grava la titularidad de vehículos matriculados. Por lo tanto, la respuesta es: anualmente.
Necesitó corrección: False


<a name="checkpoints"></a>
## 5. Checkpoints y Persistencia

In [12]:
from langgraph.checkpoint.memory import MemorySaver

# Create checkpointer
memory = MemorySaver()

# Compile with checkpointer
rag_with_memory = rag_workflow.compile(checkpointer=memory)

# Run with thread_id for session tracking
config = {"configurable": {"thread_id": "session1"}}

result = rag_with_memory.invoke({
    "messages": [HumanMessage(content="¿Qué impuestos hay?")],
    "context": "",
    "response": ""
}, config=config)

print(f"Respuesta: {result['response']}")

Respuesta: Basándome en el contexto proporcionado, puedo identificar dos tipos de impuestos mencionados:

1. **Impuesto sobre la renta o impuesto general**: Se menciona que las bonificaciones pueden reducir hasta un 90% este impuesto, aunque no se especifica su nombre exacto. Esto sugiere que hay un impuesto sobre la renta o un impuesto general que puede ser objeto de bonificaciones.

2. **Impuesto sobre Bienes Inmuebles (IBI)**: Se menciona específicamente que el IBI se paga anualmente y que su cálculo se basa en el valor catastral de la propiedad. El IBI es un impuesto local que grava la propiedad de bienes inmuebles.

Es importante destacar que el contexto no proporciona una lista exhaustiva de todos los impuestos que podrían existir, sino que solo menciona estos dos en relación con las bonificaciones y el pago anual basado en el valor catastral.


<a name="ejercicios"></a>
## 6. Ejercicios Prácticos

### Ejercicio: Crear un flujo con múltiples pasos

In [ ]:
# Exercise: Create a workflow that:
# 1. Receives a question
# 2. Classifies the question type
# 3. Retrieves relevant info
# 4. Generates response
# 5. Checks quality
# 6. Corrects if needed

In [13]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 1. Definir el Estado (State)
class FullWorkflowState(TypedDict):
    query: str
    query_type: str
    context: str
    response: str
    needs_correction: bool
    corrected_response: str

# 2. Nodo: Clasificar el tipo de pregunta
def classify_question(state: FullWorkflowState) -> FullWorkflowState:
    """Clasifica si la pregunta es sobre impuestos (requiere RAG) o general."""
    query = state["query"].lower()
    if any(word in query for word in ["ibi", "ivtm", "icio", "impuesto", "bonificación", "pagar"]):
        q_type = "taxes"
    else:
        q_type = "general"
    return {**state, "query_type": q_type}

# 3. Nodo: Recuperar información relevante (RAG)
def retrieve_info(state: FullWorkflowState) -> FullWorkflowState:
    """Recupera contexto solo si es una pregunta específica, si no, lo deja vacío."""
    if state["query_type"] == "taxes":
        docs = retriever.invoke(state["query"])
        context = "\n".join([d.page_content for d in docs])
    else:
        context = "No se requiere contexto específico. Responde de manera general."
    return {**state, "context": context}

# 4. Nodo: Generar respuesta
def generate_response(state: FullWorkflowState) -> FullWorkflowState:
    """Genera la respuesta usando el LLM y el contexto recuperado."""
    prompt = f"""Responde a la pregunta basándote en el contexto si lo hay.
    
    Contexto: {state['context']}
    Pregunta: {state['query']}
    
    Respuesta:"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

# 5. Nodo: Comprobar la calidad
def check_quality(state: FullWorkflowState) -> FullWorkflowState:
    """Verifica si la respuesta generada responde bien a la pregunta."""
    check_prompt = f"""¿La siguiente respuesta aborda correctamente la pregunta basándose en el contexto dado?
    
    Contexto: {state['context']}
    Pregunta: {state['query']}
    Respuesta: {state['response']}
    
    Responde solo 'SI' o 'NO'."""
    
    check = llm.invoke(check_prompt)
    needs_correction = "NO" in check.content.upper()
    return {**state, "needs_correction": needs_correction}

# 6. Nodo: Corregir si es necesario
def correct_response(state: FullWorkflowState) -> FullWorkflowState:
    """Aplica una corrección a la respuesta original si falló el control de calidad."""
    correct_prompt = f"""Mejora y corrige esta respuesta para que sea más precisa y útil según el contexto.
    
    Contexto: {state['context']}
    Pregunta: {state['query']}
    Respuesta original: {state['response']}
    
    Respuesta mejorada:"""
    
    corrected = llm.invoke(correct_prompt)
    return {**state, "corrected_response": corrected.content}

# Función de Enrutamiento Condicional
def route_after_check(state: FullWorkflowState) -> Literal["correct", "end"]:
    return "correct" if state["needs_correction"] else "end"

# --- CONSTRUCCIÓN DEL GRAFO ---

workflow = StateGraph(FullWorkflowState)

# Añadir Nodos
workflow.add_node("classify", classify_question)
workflow.add_node("retrieve", retrieve_info)
workflow.add_node("generate", generate_response)
workflow.add_node("check", check_quality)
workflow.add_node("correct", correct_response)

# Añadir Aristas (Edges)
workflow.add_edge(START, "classify")
workflow.add_edge("classify", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "check")

# Borde Condicional: Verifica si necesita pasar por el nodo de corrección o terminar
workflow.add_conditional_edges(
    "check",
    route_after_check,
    {"correct": "correct", "end": END}
)

workflow.add_edge("correct", END)

# Compilar
full_app = workflow.compile()
print("Flujo completo compilado ✓")

Flujo completo compilado ✓


In [14]:
# Test del flujo con una pregunta válida
inputs = {
    "query": "¿Qué grava el IVTM y el ICIO?", 
    "query_type": "", 
    "context": "", 
    "response": "", 
    "needs_correction": False, 
    "corrected_response": ""
}

result = full_app.invoke(inputs)

print(f"Tipo de consulta identificada: {result['query_type']}")
print(f"Contexto recuperado:\n{result['context']}\n")
print(f"Respuesta inicial:\n{result['response']}\n")
print(f"¿Necesitó corrección?: {result['needs_correction']}")

if result['needs_correction']:
    print(f"\nRespuesta Final Corregida:\n{result['corrected_response']}")

Tipo de consulta identificada: taxes
Contexto recuperado:
El IBI se paga anualmente basado en el valor catastral.
El IVTM grava la titularidad de vehículos matriculados.

Respuesta inicial:
El IVTM grava la titularidad de vehículos matriculados, mientras que el ICIO (Impuesto sobre Construcciones, Instalaciones y Obras) grava la realización de construcciones, instalaciones y obras.

¿Necesitó corrección?: False


## Resumen

En este notebook hemos aprendido:

1. **LangGraph**: Crear flujos como grafos de estados
2. **Condicionales**: Routing basado en resultados
3. **RAG workflow**: Retrieve → Generate
4. **Auto-corrección**: Verificar y mejorar respuestas
5. **Checkpoints**: Persistencia de sesiones

En el siguiente notebook veremos **RAG Avanzado Agentic** con flujos completos.

---

## Referencias

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)

In [ ]:
import session_info
session_info.show(html=False)